# NBA player role map

Bryan Chen · STA437H1, University of Toronto. Descriptive analysis of 2023–24 per-game NBA.com statistics. This is a role map, not an ability ranking. Run from this directory using the included CSV. DATA_SOURCE.md records its provenance.

## Data processing and dependence
Keep GP >= 10 and MIN >= 10; select 18 numeric statistics and drop incomplete rows. Preserve encoded zero percentages: a zero-attempt percentage is not measured shooting accuracy. Counts, minutes and percentages have different scales. The corner plot and correlation heatmap show dependence before PCA.

In [1]:
"""
Project 1 (NBA 2023-24) - Final polished analysis code 

Outputs:
- figures/ : publication-quality plots
- outputs/ : tables (summary stats, loadings, extremes, outliers, Mardia test)

Note: This is reproducible: run the included snapshot from this directory; replacement data require reviewing the validation assertions.
"""

from __future__ import annotations

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from scipy.stats import chi2, norm


# =========================
# 0) Config
# =========================
DATA_PATH = "nba_2023_24_stats.csv"   # <-- change if needed
FIG_DIR = "figures"
OUT_DIR = "outputs"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# Matplotlib global style (simple + readable)
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11
})

# Core stats recommended by README
STAT_COLS = [
    "MIN","PTS","FGM","FGA","FG_PCT","FG3M","FG3A","FG3_PCT",
    "FTM","FTA","FT_PCT","OREB","DREB","REB","AST","STL","BLK","TOV"
]

# A small subset for a readable cornerplot
EDA_COLS = ["PTS", "FGA", "FG3A", "REB", "AST", "TOV"]

# Filter recommended by README
MIN_GP = 10
MIN_MINUTES = 10

# Heatmap ordering (helps show block structure)
HEATMAP_ORDER = [
    # Usage / scoring volume
    "MIN","PTS","FGM","FGA","FTM","FTA","TOV","AST",
    # 3PT profile
    "FG3M","FG3A","FG3_PCT",
    # Efficiency
    "FG_PCT","FT_PCT",
    # Rebounding / defense
    "OREB","DREB","REB","STL","BLK"
]


# =========================
# Helper functions
# =========================
def save_fig(filename: str) -> None:
    path = os.path.join(FIG_DIR, filename)
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches="tight")
    print(f"[Saved] {path}")


def label_extremes_minimal(
    ax,
    df: pd.DataFrame,
    xcol: str,
    ycol: str,
    name_col: str,
    team_col: str,
    k: int = 3
) -> None:
    """
    Label only top/bottom k points by xcol and ycol (unique rows).
    Adds a small offset and a white bbox to keep labels readable.
    """
    top_x = df.nlargest(k, xcol)
    bot_x = df.nsmallest(k, xcol)
    top_y = df.nlargest(k, ycol)
    bot_y = df.nsmallest(k, ycol)

    lab = pd.concat([top_x, bot_x, top_y, bot_y]).drop_duplicates()

    # simple deterministic offsets to reduce overlap
    offsets = [(6, 6), (6, -6), (-6, 6), (-6, -6), (10, 0), (0, 10), (-10, 0), (0, -10)]

    for i, (_, r) in enumerate(lab.iterrows()):
        dx, dy = offsets[i % len(offsets)]
        ax.annotate(
            f'{r[name_col]} ({r[team_col]})',
            (r[xcol], r[ycol]),
            textcoords="offset points",
            xytext=(dx, dy),
            ha="left" if dx >= 0 else "right",
            va="bottom" if dy >= 0 else "top",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75),
            fontsize=10
        )


def corr_heatmap_annotated(corr: pd.DataFrame, title: str, filename: str,
                           decimals: int = 2, fontsize: int = 7) -> None:
    import matplotlib.pyplot as plt
    import numpy as np

    plt.figure(figsize=(12, 10))
    im = plt.imshow(corr.values, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
    cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
    cbar.set_label("Correlation")

    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
    plt.yticks(range(len(corr.index)), corr.index)

    # annotate every cell
    for i in range(corr.shape[0]):
        for j in range(corr.shape[1]):
            val = corr.values[i, j]
            plt.text(j, i, f"{val:.{decimals}f}", ha="center", va="center",
                     fontsize=fontsize, color="black")

    plt.title(title)
    plt.tight_layout()
    plt.savefig(os.path.join("figures", filename), dpi=250, bbox_inches="tight")
    plt.close()



def compute_loadings(pca: PCA) -> np.ndarray:
    """
    Loadings for standardized PCA:
      loadings = components_.T * sqrt(eigenvalues * (n - 1) / n)
    This gives variable-PC correlations with StandardScaler population scaling.
    """
    eigvals = pca.explained_variance_
    return pca.components_.T * np.sqrt(eigvals * (pca.n_samples_ - 1) / pca.n_samples_)


def plot_top_loadings(load_df: pd.DataFrame, col: str, top_n: int, title: str, filename: str) -> None:
    """
    Plot top_n variables by absolute loading for a given PC column.
    Adds a 0 reference line for sign clarity.
    """
    s = load_df[col].copy()
    top = s.reindex(s.abs().sort_values(ascending=False).head(top_n).index)
    top = top.sort_values()  # nice horizontal ordering

    plt.figure(figsize=(7.6, 5.8))
    plt.barh(top.index, top.values)
    plt.axvline(0, linewidth=1)
    plt.xlabel("Loading")
    plt.title(title)
    save_fig(filename)
    plt.close()


def biplot_pc12_sparse(
    scores_2d: np.ndarray,
    loadings_2d: np.ndarray,
    var_names: list[str],
    title: str,
    filename: str,
    loading_threshold: float = 0.30
) -> None:
    """
    Cleaner biplot: plot points + only variables with strong loadings in PC1/PC2.
    """
    plt.figure(figsize=(9.2, 7.2))
    plt.scatter(scores_2d[:, 0], scores_2d[:, 1], s=18, alpha=0.45)

    # Choose variables with strong representation in PC1/PC2
    mask = (np.abs(loadings_2d[:, 0]) >= loading_threshold) | (np.abs(loadings_2d[:, 1]) >= loading_threshold)
    idxs = np.where(mask)[0]

    # Scale arrows for visibility
    arrow_scale = 2.8
    for i in idxs:
        x, y = loadings_2d[i, 0] * arrow_scale, loadings_2d[i, 1] * arrow_scale
        plt.arrow(0, 0, x, y, head_width=0.08, length_includes_head=True)
        plt.text(x * 1.08, y * 1.08, var_names[i], fontsize=10)

    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.xlabel("PC1 score")
    plt.ylabel("PC2 score")
    plt.title(title)

    save_fig(filename)
    plt.close()


def mardia_test(X: np.ndarray) -> dict:
    """
    Mardia skewness and kurtosis (approx).
    Input X: (n, p), already standardized if you want a correlation-based MVN check.
    """
    n, p = X.shape
    Xc = X - X.mean(axis=0)

    S = np.cov(Xc, rowvar=False, bias=False)
    Sinv = np.linalg.pinv(S)

    D = Xc @ Sinv @ Xc.T  # d_ij matrix

    # Skewness: b1p = mean(d_ij^3)
    b1p = np.mean(D**3)

    # Approx chi-square
    df_chi = p * (p + 1) * (p + 2) / 6
    chi2_stat = n * b1p / 6
    p_skew = chi2.sf(chi2_stat, df=df_chi)

    # Kurtosis: b2p = mean(d_ii^2)
    d2 = np.diag(D)
    b2p = np.mean(d2**2)

    expected = p * (p + 2)
    var = (8 * p * (p + 2)) / n
    z_kurt = (b2p - expected) / np.sqrt(var)
    p_kurt = 2 * norm.sf(abs(z_kurt))

    return {
        "n": int(n),
        "p": int(p),
        "b1p_skew": float(b1p),
        "chi2_stat_skew": float(chi2_stat),
        "df_skew": float(df_chi),
        "p_value_skew": float(p_skew),
        "b2p_kurt": float(b2p),
        "z_kurt": float(z_kurt),
        "p_value_kurt": float(p_kurt),
    }


# =========================
# 1) Load + Filter + Select
# =========================
df = pd.read_csv(DATA_PATH)

# IMPORTANT: Do NOT turn pct columns into NaN; keep dataset encoding
# This keeps the full filtered sample size (typically 421 for GP>=10 & MIN>=10).
df_f = df[(df["GP"] >= MIN_GP) & (df["MIN"] >= MIN_MINUTES)].copy()

META_COLS = ["PLAYER_NAME", "TEAM_ABBREVIATION", "GP", "MIN"]
meta = df_f[META_COLS].copy()

X_raw = df_f[STAT_COLS].copy()

# If there are rare missing values, drop those rows only (should be minimal)
mask_complete = X_raw.notna().all(axis=1)
X_raw = X_raw.loc[mask_complete].copy()
meta = meta.loc[mask_complete].copy()

print("Filtered players (final n):", len(X_raw))
print("Core features (p):", len(STAT_COLS))

# Summary table
summary = X_raw.describe().T
summary.to_csv(os.path.join(OUT_DIR, "summary_stats_core18.csv"))
print(f"[Saved] {os.path.join(OUT_DIR, 'summary_stats_core18.csv')}")


# =========================
# 2) Cornerplot (subset)
# =========================
corner_df = df_f.loc[mask_complete, EDA_COLS].copy()
pd.plotting.scatter_matrix(
    corner_df,
    figsize=(10, 10),
    diagonal="hist",
    alpha=0.6
)
plt.suptitle("Cornerplot (subset of core NBA stats)", y=1.02)
save_fig("cornerplot_subset.png")
plt.close()


# =========================
# 3) Figure 1: Correlation heatmap (18 vars)
# =========================
# Reorder to emphasize blocks
order = [c for c in HEATMAP_ORDER if c in STAT_COLS]
corr = X_raw[order].corr()

corr_heatmap_annotated(
    corr,
    title="Correlation heatmap (18 core stats, filtered players)",
    filename="Fig1_corr_heatmap_core18_annotated.png"
)




Filtered players (final n): 421
Core features (p): 18
[Saved] outputs\summary_stats_core18.csv


[Saved] figures\cornerplot_subset.png


## Standardized PCA
StandardScaler centers and scales each feature using population variance. PCA uses all 18 features. Orient PC1 toward points and PC2 toward offensive rebounds so plots and prose remain consistent across sign conventions. The first two components summarize variance; they do not predict performance.

In [2]:
# =========================
# 4) PCA (Correlation PCA = z-score)
# =========================
scaler = StandardScaler()
Xz = scaler.fit_transform(X_raw.values)

pca = PCA(svd_solver="full")
Z = pca.fit_transform(Xz)

# Fix arbitrary signs consistently for components and scores.
for component, anchor in [(0, "PTS"), (1, "OREB")]:
    if pca.components_[component, STAT_COLS.index(anchor)] < 0:
        pca.components_[component] *= -1
        Z[:, component] *= -1
expl = pca.explained_variance_ratio_
cum = np.cumsum(expl)

In [3]:
# =========================
# 5) Figure 2: Scree plot (with annotations)
import os
import numpy as np
import matplotlib.pyplot as plt

K = 10
expl = pca.explained_variance_ratio_
cum = np.cumsum(expl)

pc1 = expl[0] * 100
pc2 = expl[1] * 100
cum2 = cum[1] * 100

fig, ax = plt.subplots(figsize=(9.2, 4.8))
x = np.arange(1, K + 1)

ax.plot(x, expl[:K], marker="o", label="Explained variance ratio")
ax.plot(x, cum[:K], marker="o", label="Cumulative")

ax.axvline(2, linestyle="--", linewidth=1)

ax.set_xlabel("Principal component")
ax.set_ylabel("Proportion of variance")
ax.set_title("Scree plot (standardized PCA)")

# ---- move legend OUTSIDE to the right ----
leg = ax.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    borderaxespad=0.0
)

# ---- move the annotation box LEFT inside the plot ----
txt = (
    f"PC1 = {pc1:.1f}%\n"
    f"PC2 = {pc2:.1f}%\n"
    f"Cumulative (PC1-PC2) = {cum2:.1f}%"
)
ax.text(
    0.6, 0.85, txt,
    transform=ax.transAxes,
    ha="left", va="top",
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.85),
    fontsize=12
)

# ---- save WITHOUT tight_layout() to avoid pulling legend back in ----
out_path = os.path.join("figures", "Fig2_scree_plot_v3.png")
fig.savefig(out_path, dpi=250, bbox_inches="tight", bbox_extra_artists=(leg,))
plt.close(fig)

print("Saved:", out_path)
print("PC1:", pc1)
print("PC2:", pc2)



Saved: figures\Fig2_scree_plot_v3.png
PC1: 51.18269188745851
PC2: 22.632331593995396


## Player role map
PC1 describes offensive involvement (not the formal basketball usage percentage). Positive PC2 describes interior style; negative PC2 describes perimeter style. Labels highlight score extremes, not a statistical ranking.

In [4]:
# =========================
# 6) Figure 3: Scores plot (PC1 vs PC2) - label minimal extremes + orange highlights (final)
# =========================
scores = pd.DataFrame(Z[:, :2], columns=["PC1", "PC2"], index=X_raw.index)
scores = scores.join(meta[["PLAYER_NAME", "TEAM_ABBREVIATION"]])

fig, ax = plt.subplots(figsize=(8.6, 6.4))

# Base layer: all players (subtle)
ax.scatter(scores["PC1"], scores["PC2"], s=18, alpha=0.55)

# Reference lines at origin (neutral color so they don't dominate)
ax.axhline(0, color="gray", alpha=0.5, linewidth=1)
ax.axvline(0, color="gray", alpha=0.5, linewidth=1)

ax.set_xlabel("PC1 score")
ax.set_ylabel("PC2 score")
ax.set_title("NBA player role map - standardized PCA")

# Label only a small set of extremes (k=3)
# Labels use the helper defined above; only score extremes are annotated.
label_extremes_minimal(
    ax,
    df=scores,
    xcol="PC1",
    ycol="PC2",
    name_col="PLAYER_NAME",
    team_col="TEAM_ABBREVIATION",
    k=3
)

# Highlight the same extreme points in orange (overlay)
ext = pd.concat([
    scores.nlargest(3, "PC1"),
    scores.nsmallest(3, "PC1"),
    scores.nlargest(3, "PC2"),
    scores.nsmallest(3, "PC2"),
]).drop_duplicates()

ax.scatter(
    ext["PC1"], ext["PC2"],
    s=85, alpha=0.95,
    color="tab:orange",
    edgecolors="black",
    linewidths=0.6
)

save_fig("Fig3_scores_PC1_PC2_labeled.png")
plt.close()

# Save extremes tables (top/bottom 10, useful for write-up)
ext_pc1 = pd.concat([scores.nlargest(10, "PC1"), scores.nsmallest(10, "PC1")]).drop_duplicates()
ext_pc2 = pd.concat([scores.nlargest(10, "PC2"), scores.nsmallest(10, "PC2")]).drop_duplicates()
ext_pc1.to_csv(os.path.join(OUT_DIR, "extremes_PC1.csv"), index=False)
ext_pc2.to_csv(os.path.join(OUT_DIR, "extremes_PC2.csv"), index=False)
print(f"[Saved] {os.path.join(OUT_DIR, 'extremes_PC1.csv')}")
print(f"[Saved] {os.path.join(OUT_DIR, 'extremes_PC2.csv')}")




[Saved] figures\Fig3_scores_PC1_PC2_labeled.png
[Saved] outputs\extremes_PC1.csv
[Saved] outputs\extremes_PC2.csv


## Loadings
Loadings below are variable–component correlations. A finite-sample normalization corrects the original square-root-eigenvalue formula for StandardScaler’s population variance convention; the correction is about 0.12% at n=421 and does not change PCA scores or variance explained.

In [5]:
# =========================
# 7) Loadings (top 10 by |loading|)
# =========================
loadings = compute_loadings(pca)
load_df = pd.DataFrame(loadings[:, :2], index=STAT_COLS, columns=["PC1_loading", "PC2_loading"])
load_df.to_csv(os.path.join(OUT_DIR, "loadings_PC1_PC2.csv"))
print(f"[Saved] {os.path.join(OUT_DIR, 'loadings_PC1_PC2.csv')}")

plot_top_loadings(
    load_df,
    col="PC1_loading",
    top_n=10,
    title="Variable loadings (top 10 by |loading|) - PC1",
    filename="PC1_loading_top10.png"
)

plot_top_loadings(
    load_df,
    col="PC2_loading",
    top_n=10,
    title="Variable loadings (top 10 by |loading|) - PC2",
    filename="PC2_loading_top10.png"
)


[Saved] outputs\loadings_PC1_PC2.csv
[Saved] figures\PC1_loading_top10.png


[Saved] figures\PC2_loading_top10.png


In [6]:
# =========================
# 8) Cleaner biplot (only strong loading variables)
# =========================
biplot_pc12_sparse(
    scores_2d=Z[:, :2],
    loadings_2d=loadings[:, :2],
    var_names=STAT_COLS,
    title="Biplot (PC1 & PC2): scores + strong loadings (standardized PCA)",
    filename="biplot_PC1_PC2_sparse.png",
    loading_threshold=0.30
)


[Saved] figures\biplot_PC1_PC2_sparse.png


## Diagnostics and limits
All nonzero components enter Mahalanobis distance. Highly correlated and nearly algebraically dependent box-score variables can amplify rounding noise. Chi-square thresholds are approximate exploratory references for in-sample distances, not calibrated discoveries. No player is removed. Mardia uses the original sample-covariance convention and asymptotic reference; PCA itself does not require multivariate normality.

In [7]:
# =========================
# 9) Figure 4: Outliers / MVN - Mahalanobis D^2 + robust Q-Q line + threshold + orange top3 points (FINAL + lighter frame)
# =========================

# ---- 1) Compute scaled PC scores and Mahalanobis D^2 (in PC space) ----
eigvals = pca.explained_variance_
Y = Z / np.sqrt(eigvals)          # scaled PC scores (MVN -> approx N(0, I))
D2 = np.sum(Y**2, axis=1)         # squared Mahalanobis distances

# Build outlier table with IDs + meta
out_df = meta.copy()
out_df["D2"] = D2
out_df = out_df.sort_values("D2", ascending=False)

# Save outlier tables
out_df.to_csv(os.path.join(OUT_DIR, "mahalanobis_D2_all.csv"), index=False)
out_df.head(20).to_csv(os.path.join(OUT_DIR, "top20_outliers_by_D2.csv"), index=False)
print(f"[Saved] {os.path.join(OUT_DIR, 'mahalanobis_D2_all.csv')}")
print(f"[Saved] {os.path.join(OUT_DIR, 'top20_outliers_by_D2.csv')}")

# ---- 2) Chi-square Q-Q coordinates ----
p = Y.shape[1]
D2_sorted = np.sort(D2)
q = (np.arange(1, len(D2_sorted) + 1) - 0.5) / len(D2_sorted)
theory = chi2.ppf(q, df=p)

# ---- 3) Plot ----
fig, ax = plt.subplots(figsize=(7.0, 6.8))
ax.scatter(theory, D2_sorted, s=18, alpha=0.7)

# Robust reference line using middle quantiles (10% to 90%)
lo, hi = int(0.10 * len(D2_sorted)), int(0.90 * len(D2_sorted))
x_mid = theory[lo:hi]
y_mid = D2_sorted[lo:hi]
slope, intercept = np.polyfit(x_mid, y_mid, 1)
xline = np.array([theory.min(), theory.max()])
ax.plot(xline, slope * xline + intercept)

ax.set_xlabel(r"Theoretical $\chi^2$ quantiles (df = p)")
ax.set_ylabel(r"Empirical $D^2$ quantiles")
ax.set_title(r"Mahalanobis $D^2$ Q-Q plot (MVN check / outlier screening)")

# ---- Make the frame lighter (hide top/right spines) ----
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)

# ---- 4) Add chi-square threshold line used for outlier flagging ----
thr = chi2.ppf(0.999, df=p)
ax.axhline(thr, linestyle="--", linewidth=1)
ax.text(
    theory.min(),
    thr + 3.5,
    rf"$\chi^2_{{0.999,{p}}}$ threshold",
    fontsize=10,
    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75)
)

# ---- 5) Label + highlight top 3 outliers (orange points + non-overlapping labels) ----
top3 = out_df.head(3).copy()

offsets = [(10, 10), (10, -18), (-10, 10)]  # three distinct positions
has = ["left", "left", "right"]
vas = ["bottom", "top", "bottom"]

for (_, r), (dx, dy), ha, va in zip(top3.iterrows(), offsets, has, vas):
    val = float(r["D2"])
    j = int(np.searchsorted(D2_sorted, val, side="right") - 1)
    j = max(0, min(j, len(D2_sorted) - 1))

    # highlight the point in orange
    ax.scatter(
        theory[j], D2_sorted[j],
        s=85, alpha=0.95,
        color="tab:orange",
        edgecolors="black",
        linewidths=0.6
    )

    # label the point
    ax.annotate(
        f'{r["PLAYER_NAME"]} ({r["TEAM_ABBREVIATION"]})',
        (theory[j], D2_sorted[j]),
        textcoords="offset points",
        xytext=(dx, dy),
        ha=ha, va=va,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75),
        fontsize=10
    )

save_fig("Fig4_mahalanobis_D2_QQ.png")
plt.close(fig)

# ---- 6) Flag outliers beyond chi-square threshold ----
flagged = out_df[out_df["D2"] > thr].copy()
flagged.to_csv(os.path.join(OUT_DIR, "flagged_outliers_chi2_0p999.csv"), index=False)
print(f"[Saved] {os.path.join(OUT_DIR, 'flagged_outliers_chi2_0p999.csv')}")
print("Flagged outliers count (D2 > chi2_0.999):", len(flagged))




[Saved] outputs\mahalanobis_D2_all.csv
[Saved] outputs\top20_outliers_by_D2.csv


[Saved] figures\Fig4_mahalanobis_D2_QQ.png
[Saved] outputs\flagged_outliers_chi2_0p999.csv
Flagged outliers count (D2 > chi2_0.999): 23


In [8]:
# =========================
# 10) Mardia tests 
# =========================
mardia = mardia_test(Xz)
pd.DataFrame([mardia]).to_csv(os.path.join(OUT_DIR, "mardia_test_results.csv"), index=False)
print(f"[Saved] {os.path.join(OUT_DIR, 'mardia_test_results.csv')}")
print("Mardia test results:", mardia)

print("\nDone. Figures saved in:", FIG_DIR, "and outputs in:", OUT_DIR)

[Saved] outputs\mardia_test_results.csv
Mardia test results: {'n': 421, 'p': 18, 'b1p_skew': 123.14506760067879, 'chi2_stat_skew': 8640.678909980961, 'df_skew': 1140.0, 'p_value_skew': 0.0, 'b2p_kurt': 547.9126815702216, 'z_kurt': 71.8457187982562, 'p_value_kurt': 0.0}

Done. Figures saved in: figures and outputs in: outputs


In [9]:
# Compact reproducibility evidence; validate against independent calculations.
import json, hashlib, platform, importlib.metadata
input_hash = hashlib.sha256(open(DATA_PATH, "rb").read()).hexdigest()
assert input_hash == "d604caddec43b15e638ef366e0363df7f0c6335da95c6263ee539721070db611", "Input differs from the documented snapshot"
assert df.shape == (572, 67)
assert df["PLAYER_ID"].is_unique and df["PLAYER_NAME"].is_unique
assert len(X_raw) == 421 and X_raw.shape[1] == 18
assert np.allclose(Xz.mean(axis=0), 0, atol=1e-12)
assert np.allclose(Xz.var(axis=0), 1)
assert np.allclose(loadings[:, :2], np.corrcoef(Xz.T, Z[:, :2].T)[:18, 18:])
assert np.allclose(D2, np.einsum("ij,jk,ik->i", Xz, np.linalg.pinv(np.cov(Xz, rowvar=False)), Xz))
audit = {
    "python": platform.python_version(), "platform": platform.system(),
    "packages": {k: importlib.metadata.version(k) for k in ["numpy", "pandas", "matplotlib", "scikit-learn", "scipy", "nbformat", "nbclient", "ipykernel", "jupyter-client"]},
    "input_sha256": input_hash,
    "verified_on": "2026-10-01",
    "raw_shape": list(df.shape), "filtered_n": len(X_raw), "missing_removed": int((~mask_complete).sum()),
    "zero_attempts": {k: int((df_f[k] == 0).sum()) for k in ["FGA", "FG3A", "FTA"]},
    "explained_variance": expl.tolist(), "eigenvalues": eigvals.tolist(),
    "rank": int(np.linalg.matrix_rank(Xz)), "condition_number": float(np.linalg.cond(np.cov(Xz, rowvar=False))),
    "threshold": float(thr), "flagged": len(flagged), "mardia": mardia,
    "correlations": {"PTS_FGM": float(corr.loc["PTS", "FGM"]), "PTS_FGA": float(corr.loc["PTS", "FGA"])},
    "assertions": "passed: snapshot hash and shape, unique players, sample, scaling, loading correlations, independent Mahalanobis calculation"
}
with open(os.path.join(OUT_DIR, "validation.json"), "w") as f:
    json.dump(audit, f, indent=2)
print(json.dumps(audit, indent=2))


{
  "python": "3.12.14",
  "platform": "Windows",
  "packages": {
    "numpy": "2.5.3",
    "pandas": "3.0.6",
    "matplotlib": "3.11.2",
    "scikit-learn": "1.9.1",
    "scipy": "1.18.1",
    "nbformat": "5.11.1",
    "nbclient": "0.11.0",
    "ipykernel": "7.4.0",
    "jupyter-client": "8.10.0"
  },
  "input_sha256": "d604caddec43b15e638ef366e0363df7f0c6335da95c6263ee539721070db611",
  "verified_on": "2026-10-01",
  "raw_shape": [
    572,
    67
  ],
  "filtered_n": 421,
  "missing_removed": 0,
  "zero_attempts": {
    "FGA": 0,
    "FG3A": 26,
    "FTA": 1
  },
  "explained_variance": [
    0.5118269188745851,
    0.22632331593995397,
    0.050946319461219636,
    0.043371549231188376,
    0.03638084563567958,
    0.029283429945123646,
    0.027609187661299186,
    0.025737130192570477,
    0.01966500308722237,
    0.008275747476110939,
    0.007597231229561396,
    0.006728362064325695,
    0.005232785590465462,
    0.0005518338305508646,
    0.00029377480659800156,
    0.000156

## Results and interpretation

The included snapshot yields **421 players** and **18 features**. PC1 explains **51.18%**, PC2 **22.63%**, and together they explain **73.82%** of standardized variance. Only the first two eigenvalues exceed one.

- **PC1: offensive involvement.** Positive correlations include PTS (0.978), FGM (0.971), FGA (0.961) and MIN (0.921).
- **PC2: interior versus perimeter style.** OREB (0.866), FG_PCT (0.729), REB (0.687) and BLK (0.653) oppose FG3A (-0.649) and FG3M (-0.641).
- **Diagnostics:** the exploratory chi-square(18) 0.999 cutoff is 42.312 and flags 23 profiles. Approximate Mardia skewness is 8640.68 (df=1140), kurtosis z is 71.85, both p < 1e-16. Printed p=0 reflects numerical underflow. Flagged players are retained.

![Player role map](figures/Fig3_scores_PC1_PC2_labeled.png)

![Explained variance](figures/Fig2_scree_plot_v3.png)

![Correlation heatmap](figures/Fig1_corr_heatmap_core18_annotated.png)

This single-season role map is descriptive. Minutes, team context, redundant box-score measures, percentage encoding and the GP/MIN selection affect interpretation. The covariance condition number is about 113,852, so full-dimensional distance rankings can be sensitive to rounding. These axes are not ability rankings, causal effects or performance predictions. PCA does not require multivariate normality.

See README.md for reproduction instructions and REPORT_VERIFICATION.md for the small loading normalization correction accompanying the unchanged original PDF.
